# Radio Survey Pipelines II
## Transfer learning and evidence on an imbalanced LoTSS sample

**90 minutes · pretrained CNN or Vision Transformer · results and errors**

This notebook consumes the data decision from Session I. It supports one
controlled configuration at a time, with optional comparisons if the tutor
chooses to organise them.

### Guiding question

> How do the pretrained representation and imbalance strategy affect
minority-class performance and failure modes?

The training machinery is supplied. This is a guided experiment in reading
model evidence: choose one controlled configuration, optionally compare it
with another controlled run, and diagnose results at class level rather than
from a single accuracy number. Cells with controls can be rerun; alter one
setting at a time so that a comparison has a clear interpretation.

In [ ]:
#@title Set up a clean Hugging Face GPU runtime
# This cell is designed for a fresh Colab runtime. It removes older
# Transformers files before installing one internally consistent release.
# If an earlier version of this notebook was run, use Runtime > Disconnect
# and delete runtime, then run this cell once.
%pip -q uninstall -y transformers tokenizers
%pip -q install --no-cache-dir -U astropy "transformers==5.16.1" "accelerate>=1.2,<2" "huggingface-hub>=1.23,<2"

from pathlib import Path
import hashlib, json, logging, os, random, tarfile
# All selected checkpoints are public: do not look for a personal HF token.
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from astropy.io import fits
from astropy.visualization import ZScaleInterval
from scipy.ndimage import gaussian_filter, shift
from google.colab import drive
import torch
from torch import nn
from torch.utils.data import Dataset, WeightedRandomSampler
from torchvision import transforms
import transformers
import huggingface_hub
from transformers import (
    AutoConfig, AutoImageProcessor, AutoModelForImageClassification,
    Trainer, TrainingArguments,
)
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, f1_score,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if not torch.cuda.is_available():
    raise RuntimeError("Choose Runtime, Change runtime type, then T4 GPU.")
# Public checkpoints do not require login.  Silence the non-actionable
# public-download rate-limit reminder in a classroom setting.
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
print("GPU:", torch.cuda.get_device_name(0))
print(f"Transformers {transformers.__version__} | Hugging Face Hub {huggingface_hub.__version__}")
drive.mount("/content/drive")
course_dir = Path("/content/drive/MyDrive/Granada School")
course_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
#@title Locate and verify the LoTSS classroom archive
# Before opening this notebook, drag the supplied archive into:
# My Drive/Granada School/Granada_School_LoTSS_raw.tar.gz
ARCHIVE_FILENAME = "Granada_School_LoTSS_raw.tar.gz" #@param {type:"string"}
EXPECTED_SHA256 = "970d2fdcad11fbbcffd8c74fca8e2dc491adb253bd92e2913e5511e3d3046211"

archive_path = course_dir / ARCHIVE_FILENAME
data_root = Path("/content/granada_lotss")

if len(EXPECTED_SHA256) != 64:
    raise RuntimeError("The published notebook must contain the archive checksum.")

if not archive_path.exists():
    raise FileNotFoundError(
        "The supplied archive was not found. Drag "
        f"{ARCHIVE_FILENAME} into My Drive/Granada School, then rerun this cell."
    )
print(f"Found classroom archive in Google Drive: {archive_path.name}")

hasher = hashlib.sha256()
with archive_path.open("rb") as stream:
    for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
        hasher.update(chunk)
digest = hasher.hexdigest()
if digest != EXPECTED_SHA256:
    raise RuntimeError(f"Archive checksum mismatch: {digest}")

if not (data_root / "manifest.csv").is_file():
    print("Extracting the verified archive...")
    with tarfile.open(archive_path, "r:gz") as archive:
        archive.extractall("/content", filter="data")

manifest = pd.read_csv(data_root / "manifest.csv")
expected_counts = {
    "FRI": 2349,
    "FRII": 5660,
    "Hybrid": 337,
    "Spiral": 182,
    "Relaxed double": 277,
}
assert len(manifest) == 8805
assert manifest.source_id.nunique() == 8805
assert manifest.label.value_counts().to_dict() == expected_counts
assert all((data_root / path).is_file() for path in manifest.fits_path)
print("Verified 8,805 raw FITS cutouts.")
display(manifest.head(3))

## Checkpoint 1 — Reuse the Session I decision (8 minutes)

The contract saved in Google Drive by Session I is checked before training.
This is a small but important reproducibility device: the model sees the
same scaling and augmentation policy that was examined visually, rather than
an accidental new preprocessing route.

**Investigate.** Read the displayed contract before proceeding. Confirm the
five class names, the selected scaling method and the training-only
augmentation policy. If any value differs from the experiment intended for
your intended run, return to the relevant Session I control, rerun its export cell,
and then rerun this one. Do not edit the JSON by hand.

**What the code does.** The cell opens the JSON file saved in Google Drive,
checks a small set of non-negotiable fields, and checks that scaling and
augmentation values are in the supported lists. These checks fail early if
a contract is missing or incompatible, rather than silently training a
different pipeline from the one that was inspected.

In [ ]:
contract_path = course_dir / "granada_lotss_pipeline_contract.json"
if not contract_path.is_file():
    raise RuntimeError(
        "The Session I pipeline contract is missing from Google Drive. "
        "Complete and export Checkpoint 5 in Session I first."
    )
contract = json.loads(contract_path.read_text())

required = {
    "schema": "granada-lotss-pipeline-1.1",
    "cohort_rows": 8805,
    "source_channels": 1,
    "model_channels": 3,
    "image_size": 224,
}
for key, expected in required.items():
    if contract.get(key) != expected:
        raise ValueError(f"Contract mismatch for {key}: {contract.get(key)!r}")
if contract.get("scaling") not in {"zscale", "percentile", "asinh", "minmax"}:
    raise ValueError("Session I contains an unsupported scaling choice.")
augmentation_policy = contract.get("train_augmentation")
required_augmentations = {
    "d4", "translation", "white_noise", "correlated_noise", "beam_smoothing",
    "flux_scale",
}
if not isinstance(augmentation_policy, dict) or set(augmentation_policy) != required_augmentations:
    raise ValueError("Session I contains an unsupported augmentation policy.")
display(contract)

## Checkpoint 2 — Inspect the supplied lazy data route (12 minutes)

Images are decoded only when a batch requests them. Augmentation is attached
only to the training dataset.

Lazy loading avoids placing the full FITS collection in memory or converting
it to a second image format. The primary array remains the source of truth.
Session I's scaling choice is applied independently to each requested image.
A single greyscale channel then becomes three identical channels because the
pretrained weights expect the ImageNet tensor interface. ImageNet
normalisation does not turn the data into natural-colour imagery; it places
the repeated channels on the numerical scale seen during pretraining.

**Investigate.** Run the cell once and read the reported tensor shape:
`(3, 224, 224)` means three channels at the spatial size expected by the
pretrained networks. The two equality statements distinguish repetition of
the greyscale image from later channel-specific normalisation. The data route
is intentionally fixed here; its scientific alternatives were explored in
Session I, so this section documents rather than reopens those choices.

**What the code does.** `LoTSSDataset` stores manifest rows rather than the
image arrays themselves. Its `__getitem__` method reads and cleans a FITS
array only when a loader requests one item, scales it using the contract,
converts it to an 8-bit greyscale PIL image, then resizes, repeats and
normalises channels through torchvision transforms. The three dataset
objects share this route, but only the training transform includes the
selected random augmentation.

In [ ]:
zscale = ZScaleInterval()
class_order = contract["class_order"]
label_to_id = {name: number for number, name in enumerate(class_order)}

# The processor configuration is read from the selected HF checkpoint later.
# These common transforms keep the FITS route identical across model choices.
common_image_size = 224

def scale_array(image, method):
    if method == "zscale":
        return np.asarray(zscale(image), dtype=np.float32)
    low, high = np.percentile(image, [0.5, 99.5])
    if method == "minmax":
        low, high = np.min(image), np.max(image)
    if high <= low:
        return np.zeros_like(image, dtype=np.float32)
    scaled = np.clip((image - low) / (high - low), 0, 1)
    if method == "asinh":
        softening = 0.08
        scaled = np.arcsinh(scaled / softening) / np.arcsinh(1 / softening)
    return np.asarray(scaled, dtype=np.float32)

def apply_training_augmentations(image):
    def active(name):
        return (augmentation_policy[name]["enabled"] and
                np.random.random() < np.clip(augmentation_policy[name]["probability"], 0.0, 1.0))

    view = image.copy()
    if active("d4"):
        view = np.rot90(view, int(np.random.randint(0, 4)))
        if np.random.random() < 0.5:
            view = np.fliplr(view)
    if active("translation"):
        limit = max(0, int(augmentation_policy["translation"]["max_pixels"]))
        view = shift(view, shift=np.random.randint(-limit, limit + 1, size=2),
                     order=1, mode="constant", cval=0.0)
    if active("white_noise"):
        view = view + np.random.normal(
            0.0, max(0.0, augmentation_policy["white_noise"]["sigma"]), view.shape
        )
    if active("correlated_noise"):
        noise = gaussian_filter(
            np.random.normal(size=view.shape),
            sigma=max(0.0, augmentation_policy["correlated_noise"]["correlation_length_pixels"]),
        )
        noise = noise / max(float(noise.std()), 1e-6)
        view = view + noise * max(0.0, augmentation_policy["correlated_noise"]["sigma"])
    if active("beam_smoothing"):
        view = gaussian_filter(
            view, sigma=max(0.0, augmentation_policy["beam_smoothing"]["sigma_pixels"])
        )
    if active("flux_scale"):
        fraction = max(0.0, augmentation_policy["flux_scale"]["max_fraction"])
        view = view * np.random.uniform(max(0.0, 1.0 - fraction), 1.0 + fraction)
    return np.clip(view, 0.0, 1.0).astype(np.float32)

# The transform is configured after the HF image processor is loaded.
train_transform = None
evaluation_transform = None

class LoTSSFITSDataset(Dataset):
    """Local FITS decoding plus the declared Session I preprocessing."""
    def __init__(self, frame, transform, augment=False):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform
        self.augment = augment

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        image = fits.getdata(data_root / row.fits_path).astype(np.float32)
        image = np.nan_to_num(image, nan=0.0, posinf=0.0, neginf=0.0)
        image = scale_array(image, contract["scaling"])
        if self.augment:
            image = apply_training_augmentations(image)
        image = Image.fromarray(np.uint8(np.clip(image, 0, 1) * 255)).convert("RGB")
        return {"pixel_values": self.transform(image), "labels": label_to_id[row.label]}

frames = {
    split: manifest[manifest.split == split].copy().reset_index(drop=True)
    for split in ["train", "validation", "test"]
}

### Reading the tensor diagnostic

Before normalisation, all three channels are identical copies of one scaled
radio image. ImageNet normalisation uses a different mean and standard
deviation for each channel, so their numerical values subsequently differ;
no artificial colour has been created. The same normalisation is retained
because the pretrained weights expect that numerical convention. Validation
and test transforms are deterministic: random changes there would mix model
performance with a changing measurement procedure.

## Checkpoint 3 — Choose one controlled experiment (10 minutes)

The choices are deliberately small:
ResNet-18 is the fast reference CNN; MobileNetV2 is an efficient CNN;
ConvNeXt-Tiny is a modern convolutional architecture; and ViT-Base/16 is the
transformer route. Each checkpoint is downloaded only if selected.

ResNet, MobileNet and ConvNeXt build image representations with convolutional
operations and progressively wider receptive fields. ViT divides the image
into 16 by 16 patches and relates learned patch descriptors through attention.
All begin with ImageNet weights, not radio images. This deliberate domain
mismatch makes transfer learning interesting, but limits any conclusion
about the best architecture for LoTSS.

**Investigate.** First run the reference setting, `resnet18` with
`weighted_loss`. For a meaningful second run, alter one dropdown only:
change `BACKBONE` to compare representation families, or change `IMBALANCE`
to compare how rare classes are treated. Keep `EPOCHS` and `BATCH_SIZE`
fixed unless the runtime demands a smaller batch. The printed configuration
identifies the run if you later compare configurations.

**What the code does.** The dropdowns set plain configuration variables;
they do not start training. `BACKBONE` later selects one named Hugging Face
checkpoint, and `IMBALANCE` later selects a loss or sampler. `EPOCHS`
controls the number of full training passes, while `BATCH_SIZE` controls how
many FITS images are processed before one optimiser update.

In [ ]:
BACKBONE = "resnet18" #@param ["resnet18", "mobilenet_v2", "convnext_tiny", "vit_base"]
IMBALANCE = "weighted_loss" #@param ["none", "weighted_loss", "weighted_sampler"]
EPOCHS = 3 #@param {type:"integer"}
BATCH_SIZE = 64 #@param {type:"integer"}

print("Configuration:", BACKBONE, "|", IMBALANCE,
      "|", EPOCHS, "epochs | batch size", BATCH_SIZE)

## Checkpoint 4 — Load a Hugging Face model and audit the transfer boundary (10 minutes)

A pretrained backbone is loaded and only the new five-class head is trained.
This is a controlled head-only transfer experiment, not a definitive
architecture benchmark.

**Investigate.** Run this cell after choosing a backbone and look at the
trainable fraction. It should be very small because only the final head is
unfrozen. The output shape should be `(1, 5)`: one set of five unnormalised
class scores for one image. If you change the backbone dropdown, rerun this
cell and all following training cells so that the model, processor and
Trainer belong to one configuration.

**What the code does.** Hugging Face loads a checkpoint-specific image
processor and `AutoModelForImageClassification`. The model head is replaced
with five outputs, every parameter is frozen, and only the standard
`classifier` head is unfrozen. The final lines count parameters and run one
forward pass as a shape check before `Trainer` takes over optimisation.

In [ ]:
number_of_classes = len(class_order)
checkpoints = {
    "resnet18": "microsoft/resnet-18",
    "mobilenet_v2": "google/mobilenet_v2_1.0_224",
    "convnext_tiny": "facebook/convnext-tiny-224",
    "vit_base": "google/vit-base-patch16-224",
}
checkpoint = checkpoints[BACKBONE]
image_processor = AutoImageProcessor.from_pretrained(
    checkpoint, token=False, backend="torchvision",
)
image_mean, image_std = image_processor.image_mean, image_processor.image_std
train_transform = transforms.Compose([
    transforms.Resize((common_image_size, common_image_size), antialias=True),
    transforms.ToTensor(),
    transforms.Normalize(image_mean, image_std),
])
evaluation_transform = transforms.Compose([
    transforms.Resize((common_image_size, common_image_size), antialias=True),
    transforms.ToTensor(),
    transforms.Normalize(image_mean, image_std),
])
datasets = {
    "train": LoTSSFITSDataset(frames["train"], train_transform, augment=True),
    "validation": LoTSSFITSDataset(frames["validation"], evaluation_transform),
    "test": LoTSSFITSDataset(frames["test"], evaluation_transform),
}
id2label = dict(enumerate(class_order))
model_config = AutoConfig.from_pretrained(checkpoint, token=False)
model_config.num_labels = number_of_classes
model_config.id2label = id2label
model_config.label2id = label_to_id
transformers.utils.logging.set_verbosity_error()
model = AutoModelForImageClassification.from_pretrained(
    checkpoint, config=model_config, ignore_mismatched_sizes=True, token=False,
)
transformers.utils.logging.set_verbosity_warning()
print(
    f"Loaded {BACKBONE} pretrained on ImageNet; replaced its original "
    f"classifier with a newly initialised {number_of_classes}-class radio-morphology head."
)

for parameter in model.parameters():
    parameter.requires_grad = False
for parameter in model.classifier.parameters():
    parameter.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} of {total:,}")
print(f"Trainable fraction: {100 * trainable / total:.4f}%")

sample = datasets["train"][0]
print("Input tensor:", tuple(sample["pixel_values"].shape), sample["pixel_values"].dtype)
mean_tensor = torch.tensor(image_mean)[:, None, None]
std_tensor = torch.tensor(image_std)[:, None, None]
before_normalisation = sample["pixel_values"] * std_tensor + mean_tensor
print("Repeated greyscale channels equal before normalisation:",
      torch.allclose(before_normalisation[0], before_normalisation[1], atol=1e-5) and
      torch.allclose(before_normalisation[1], before_normalisation[2], atol=1e-5))
print("Channels equal after channel-specific normalisation:",
      torch.allclose(sample["pixel_values"][0], sample["pixel_values"][1]) and
      torch.allclose(sample["pixel_values"][1], sample["pixel_values"][2]))
with torch.no_grad():
    output = model(pixel_values=sample["pixel_values"].unsqueeze(0))
print("Output shape:", tuple(output.logits.shape))

### What is being transferred?

The frozen backbone supplies image features shaped by ImageNet pretraining:
edges, textures, spatial arrangements and, for the ViT, patch relations. The
new linear head can reweight those features for five labels, but cannot alter
the representation itself. This makes the experiment fast and controlled;
it also means weak minority performance may reflect a radio-to-natural-image
domain mismatch rather than an intrinsic limit of the architecture.

## Checkpoint 5 — Configure imbalance handling (8 minutes)

With natural loss, a source contributes one ordinary cross-entropy term.
Weighted loss preserves the rows seen in an epoch but increases the cost of
mistakes in rare classes. Weighted sampling draws rare examples more often,
with replacement, so it changes the empirical training distribution and can
repeat a small class many times. Validation and test retain the catalogue
prevalence under all three choices.

**Investigate.** Read the displayed table before choosing a strategy. Its
`train count` column is the actual catalogue imbalance; its `loss weight`
column is inversely related to that count. With the reference
`weighted_loss`, every training source is still visited once per epoch but
rare-class mistakes matter more. With `weighted_sampler`, inspect whether
minority recall improves and whether precision or majority recall changes.

**What the code does.** `np.bincount` counts integer labels in the training
split. The inverse-frequency formula creates one weight per class. The
`weighted_loss` branch passes those weights to cross-entropy; the
`weighted_sampler` branch supplies a `WeightedRandomSampler` to `Trainer`.
Its validation and test loaders preserve their fixed source-level split.

In [ ]:
train_label_ids = frames["train"].label.map(label_to_id).to_numpy()
counts = np.bincount(train_label_ids, minlength=number_of_classes)
weights = len(train_label_ids) / (number_of_classes * counts)
class_weights = torch.tensor(weights, dtype=torch.float32)
sample_weights = torch.tensor(weights[train_label_ids], dtype=torch.double)
display(pd.DataFrame({"train count": counts, "loss weight": class_weights},
                     index=class_order))

### Reading the imbalance choice

`weighted_loss` keeps the catalogue's training rows but makes a rare-class
error more expensive. `weighted_sampler` changes the composition of batches
by drawing rare sources more often, so repetitions are expected. The held-out
loaders do neither: they are measurements on the natural curated split, not
interventions designed to make a score look balanced.

## Checkpoint 6 — Train and select on validation macro-F1 (20 minutes)

Three epochs are a classroom comparison, not a publishable optimum. The
validation split, not the test split, decides which epoch's weights are kept.

**Investigate.** Run the cell once and follow the printed record at each
epoch. Training loss should normally fall, but select the run by validation
macro-F1, not by loss and not by test performance. If comparing a second
configuration, change a single earlier control, rerun the model and loader
cells, then run training again. The small epoch count is intentional: it
leaves time to inspect failure modes rather than treating a long runtime as
evidence of scientific quality.

**What the code does.** `Trainer` now owns the batch loop, backpropagation,
optimiser, validation scheduling, metric logging and best-checkpoint reload.
Our small subclass supplies the optional weighted cross-entropy; a second
subclass supplies the optional weighted training sampler. `compute_metrics`
converts HF logits into the same accuracy, macro-F1 and balanced-accuracy
evidence used throughout the workshop.

In [ ]:
def compute_metrics(evaluation):
    logits, true = evaluation
    predicted = np.asarray(logits).argmax(axis=1)
    return {
        "accuracy": float(accuracy_score(true, predicted)),
        "macro_f1": float(f1_score(true, predicted, average="macro", zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(true, predicted)),
    }

class LoTSSTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        weight = None if self.class_weights is None else self.class_weights.to(outputs.logits.device)
        loss = nn.functional.cross_entropy(outputs.logits, labels, weight=weight)
        return (loss, outputs) if return_outputs else loss

class WeightedSamplerLoTSSTrainer(LoTSSTrainer):
    def _get_train_sampler(self):
        return WeightedRandomSampler(
            sample_weights, len(sample_weights), replacement=True,
            generator=torch.Generator().manual_seed(SEED),
        )

training_args = TrainingArguments(
    output_dir="/content/granada_hf_runs",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=1e-3,
    weight_decay=1e-4,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=1,
    report_to=[],
    remove_unused_columns=False,
    dataloader_num_workers=2,
    fp16=True,
    seed=SEED,
)
trainer_class = WeightedSamplerLoTSSTrainer if IMBALANCE == "weighted_sampler" else LoTSSTrainer
trainer = trainer_class(
    model=model,
    args=training_args,
    train_dataset=datasets["train"],
    eval_dataset=datasets["validation"],
    compute_metrics=compute_metrics,
    class_weights=class_weights if IMBALANCE == "weighted_loss" else None,
)
train_result = trainer.train()
validation_metrics = trainer.evaluate()
best_validation_macro_f1 = float(validation_metrics["eval_macro_f1"])

# The Trainer stores training and validation events on separate rows.  Build
# one readable row per epoch rather than displaying that internal event log.
epoch_summary = {}
for event in trainer.state.log_history:
    epoch = event.get("epoch")
    if epoch is None:
        continue
    epoch = int(round(epoch))
    row = epoch_summary.setdefault(epoch, {"Epoch": epoch})
    if "loss" in event:
        row["Training Loss"] = float(event["loss"])
    if "eval_loss" in event:
        row["Validation Loss"] = float(event["eval_loss"])
        row["Accuracy"] = float(event["eval_accuracy"])
        row["Macro F1"] = float(event["eval_macro_f1"])
        row["Balanced Accuracy"] = float(event["eval_balanced_accuracy"])

epoch_table = pd.DataFrame(
    [epoch_summary[epoch] for epoch in sorted(epoch_summary)]
)
display(epoch_table.round(4))

## Checkpoint 7 — Evaluate once and read class-level evidence (14 minutes)

Accuracy measures the fraction of correct predictions and therefore follows
the majority class closely. Macro-F1 computes an F1 score for each class and
averages the five values equally. Balanced accuracy averages the five
recalls. The confusion matrix adds direction: a Hybrid classified as FRII
is a different scientific failure from an FRII classified as Hybrid.

**Investigate.** Run this evaluation only after training has selected its
best validation epoch. Read the class table before the headline scores:
compare support, recall, precision and F1 for each morphology. In the matrix,
rows are true labels and columns are predictions, so off-diagonal structure
has a direction. The final error-gallery controls allow you to hide the
images or change how many high-confidence mistakes are shown without
changing the measured test scores.

**What the code does.** The cell calls `Trainer.predict` once on the
untouched test dataset, then uses scikit-learn functions to calculate summary
metrics, a per-class classification report and a count-based confusion
matrix. The heatmap is a display of that fixed matrix; it does not normalise
rows or alter the predictions.

In [ ]:
test_output = trainer.predict(datasets["test"])
test_logits = np.asarray(test_output.predictions)
test_true = np.asarray(test_output.label_ids)
test_pred = test_logits.argmax(axis=1)
test_probability = torch.softmax(torch.tensor(test_logits), dim=1).numpy()
test_sources = frames["test"].source_id.to_numpy()
test_scores = compute_metrics((test_logits, test_true))
display(test_scores)

report = pd.DataFrame(classification_report(
    test_true, test_pred, target_names=class_order,
    output_dict=True, zero_division=0,
)).T
display(report.loc[class_order, ["precision", "recall", "f1-score", "support"]])

matrix = confusion_matrix(
    test_true, test_pred, labels=range(number_of_classes)
)
plt.figure(figsize=(7, 6))
sns.heatmap(
    matrix, annot=True, fmt="d", cmap="Blues",
    xticklabels=class_order, yticklabels=class_order,
)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title(f"{BACKBONE} with {IMBALANCE}")
plt.tight_layout()

In [ ]:
SHOW_CONFIDENT_ERROR_GALLERY = True #@param {type:"boolean"}
NUMBER_OF_ERRORS_TO_SHOW = 8 #@param {type:"integer"}

confidence = test_probability.max(axis=1)
mistakes = np.where(test_true != test_pred)[0]
selected_errors = mistakes[np.argsort(confidence[mistakes])[-NUMBER_OF_ERRORS_TO_SHOW:]][::-1]

if SHOW_CONFIDENT_ERROR_GALLERY and len(selected_errors):
    columns = 4
    rows = int(np.ceil(len(selected_errors) / columns))
    fig, axes = plt.subplots(rows, columns, figsize=(14, 3.5 * rows))
    for axis, position in zip(np.ravel(axes), selected_errors):
        source = test_sources[position]
        row = frames["test"][frames["test"].source_id == source].iloc[0]
        image = fits.getdata(data_root / row.fits_path).astype(np.float32)
        image = np.nan_to_num(image, nan=0.0, posinf=0.0, neginf=0.0)
        axis.imshow(
            scale_array(image, contract["scaling"]),
            origin="lower", cmap="gray", vmin=0, vmax=1
        )
        axis.set_title(
            f"true: {class_order[test_true[position]]}\n"
            f"pred: {class_order[test_pred[position]]} "
            f"({confidence[position]:.2f})",
            fontsize=9,
        )
        axis.axis("off")
    for axis in np.ravel(axes)[len(selected_errors):]:
        axis.axis("off")
    plt.tight_layout()
elif SHOW_CONFIDENT_ERROR_GALLERY:
    print("No test-set errors were found for this configuration.")

### Reading errors as astronomical evidence

Read the confusion matrix row by row: each row asks which label a true class
becomes when it is missed. Then inspect the high-confidence errors. A
confident error may indicate a shortcut in the representation, a cutout that
omits context, a borderline morphology, or disagreement embedded in the
catalogue label. It is evidence for a follow-up investigation, not by itself
proof that the label is wrong.

When comparing runs, compare macro-F1 and every class recall before
comparing accuracy. A useful next experiment changes one factor only: for
example the imbalance strategy at fixed backbone, or the backbone at fixed
scaling and augmentation.

**What the code does.** The gallery ranks only misclassified test sources by
the largest softmax probability, then retrieves and displays their original
FITS cutouts with the fixed Session I scaling. `SHOW_CONFIDENT_ERROR_GALLERY`
controls only the display. `NUMBER_OF_ERRORS_TO_SHOW` changes the number of
ranked examples, not the model or the test metrics already calculated.

## Result memo and optional comparison (8 minutes)

Save one compact record. If you run a second controlled configuration, compare
it with this record. Do not
call the result a CNN-versus-ViT benchmark: the aim is to connect a controlled
modelling choice to class-level evidence.

**What the code does.** The final cell saves the configuration, parameter
count, best validation macro-F1, one test-score set, every class recall and
the Session I pipeline contract as a compact JSON result in Google Drive.
It does not retrain or reevaluate the model. This makes repeated results
comparable while retaining the preprocessing provenance needed to interpret
a difference.

In [ ]:
result = {
    "backbone": BACKBONE,
    "imbalance_strategy": IMBALANCE,
    "epochs": EPOCHS,
    "trainable_parameters": trainable,
    "hf_checkpoint": checkpoint,
    "best_validation_macro_f1": best_validation_macro_f1,
    "test_scores": {name: float(value) for name, value in test_scores.items()},
    "per_class_recall": {
        name: float(report.loc[name, "recall"]) for name in class_order
    },
    "pipeline_contract": contract,
}
result_path = course_dir / "granada_lotss_result.json"
result_path.write_text(
    json.dumps(result, indent=2) + "\n"
)
display(result)
print("Saved:", result_path)